In [0]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
from datasets import load_dataset

model_id = "openai/whisper-large-v3"
device = "cuda:0"
torch_dtype = torch.bfloat16

model = AutoModelForSpeechSeq2Seq.from_pretrained(
  model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)
model.to(device)

processor = AutoProcessor.from_pretrained(model_id)

pipe = pipeline(
  "automatic-speech-recognition",
  model=model,
  tokenizer=processor.tokenizer,
  feature_extractor=processor.feature_extractor,
  max_new_tokens=128,
  chunk_length_s=30,
  batch_size=16,
  return_timestamps=True,
  torch_dtype=torch_dtype,
  device=device,
)

In [0]:
%fs ls /tmp/sean.owen@databricks.com/

In [0]:
transcript = pipe('/dbfs/tmp/sean.owen@databricks.com/HSBC_Canada_Announcer_7.8.19_updated.mp3', max_new_tokens=1024)

In [0]:
transcript

In [0]:
import torch
import gc
del pipe
gc.collect()
torch.cuda.empty_cache()

In [0]:
summarizer = pipeline("summarization", model="facebook/bart-large-cnn", device="cuda:0")

In [0]:
from langchain.text_splitter import CharacterTextSplitter

text_splitter = CharacterTextSplitter(separator=". ", chunk_size=4000)
texts = text_splitter.split_text(transcript)

In [0]:
for t in texts:
  print(summarizer(t)[0]['summary_text'])
  print()